# AgenticRAG-Bench — Week 6
## D6: Difficulty Interaction Collapse

**Goal:** Prove the agent's accuracy degrades **super-linearly** when difficulty axes compound.

**Method:** Run the same 50 questions under 5 controlled difficulty conditions:

| Condition | RC (hops) | Distractors | Description |
|---|---|---|---|
| A | 1 | 6 | Easy baseline: single-hop, low noise |
| B | 2 | 6 | Hard reasoning only |
| C | 1 | 18 | Hard retrieval only |
| D | 2 | 12 | Medium both |
| E | 2 | 18 | Hard both (Week 5 baseline) |

**What does NOT change:** Agent (LangGraph ReAct + Llama 3.1 8B), retrieval (hybrid BM25+FAISS),
embeddings (nomic-embed-text), k=3, system prompt ON.

**Design notes:**
- Minimum 6 distractors ensures BM25 has enough documents for meaningful IDF scores
- 1-hop variants use decomposition step 2 with step 1's answer substituted as entity
- Supporting paragraph selection matches the paragraph actually containing the answer

---


## Cell 0 — Environment + imports

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank_bm25', 'matplotlib'])

import os
import json
import time
import math
import re
import copy
import random
import signal
from pathlib import Path
from difflib import SequenceMatcher
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
os.environ["RAGAS_DO_NOT_PARALLELIZE"] = "true"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
load_dotenv(PROJECT_ROOT / ".env")

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)

import requests
r = requests.get("http://localhost:11434/api/tags", timeout=3)
models = [m['name'] for m in r.json().get('models', [])]
print("Ollama models:", models)

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
from rank_bm25 import BM25Okapi
import numpy as np

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for notebook
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
from matplotlib import patheffects

Path(PROJECT_ROOT / "data/questions").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "data/trajectories").mkdir(parents=True, exist_ok=True)
Path(PROJECT_ROOT / "notes").mkdir(parents=True, exist_ok=True)

print("All imports OK")


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


Project root: /Users/idhantsingh/Desktop/agenticrag-bench
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]
Ollama models: ['nomic-embed-text:latest', 'llama3.1:8b']
All imports OK


---
## Cell 1 — Metric classes (D1, D2, D3, D5)

Unchanged from Week 5.

In [2]:
class D1_AnswerCorrectness:
    name = "D1_answer_correctness"
    STOPWORDS = {"the","a","an","is","was","of","in","to","and","or","it","its","be","been","being","have","has","had","by","at","from","with","that","this","are"}
    LEGAL_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","gmbh","company"}
    def _normalize(self, text):
        text = text.lower().strip()
        text = re.sub(r'[^\w\s]', ' ', text)
        words = [w for w in text.split() if w not in self.LEGAL_SUFFIXES]
        return ' '.join(words)
    def _meaningful_words(self, text):
        text = re.sub(r'[^\w\s]', ' ', text)
        return {w for w in text.split() if w not in self.STOPWORDS and w not in self.LEGAL_SUFFIXES and len(w) > 1}
    def _is_degenerate(self, predicted):
        p = predicted.strip()
        return (p.startswith('{') and 'lookup_knowledge_base' in p) or p.lower().startswith('sorry') or len(p) < 5
    def score(self, predicted, ground_truth):
        degenerate = self._is_degenerate(predicted)
        if degenerate:
            return {"exact_match": False, "near_match": False, "token_recall": 0.0, "seq_sim": 0.0, "score": 0.0, "degenerate_output": True}
        pred_raw = predicted.lower().strip()
        truth_raw = ground_truth.lower().strip()
        exact_raw = truth_raw in pred_raw
        pred_norm = self._normalize(predicted)
        truth_norm = self._normalize(ground_truth)
        if ' ' not in truth_norm:
            exact_norm = pred_norm == truth_norm or pred_norm.startswith(truth_norm + ' ') or (' is ' + truth_norm) in pred_norm or (' was ' + truth_norm) in pred_norm
        else:
            exact_norm = truth_norm in pred_norm
        exact_match = exact_raw or exact_norm
        pred_words = self._meaningful_words(pred_norm)
        truth_words = self._meaningful_words(truth_norm)
        recall = len(pred_words & truth_words) / len(truth_words) if truth_words else (1.0 if exact_match else 0.0)
        seq = SequenceMatcher(None, truth_norm, pred_norm[:len(truth_norm)*3]).ratio()
        near = (recall >= 0.6) or (seq >= 0.7 and recall >= 0.4)
        if exact_match: score = 1.0
        elif near: score = round(0.5 + recall * 0.5, 3)
        else:
            intersection = pred_words & truth_words
            union = pred_words | truth_words
            score = round(len(intersection)/len(union), 3) if union else 0.0
        return {"exact_match": exact_match, "near_match": near, "token_recall": round(recall,3), "seq_sim": round(seq,3), "score": score, "degenerate_output": False}

class D2_RetrievalStepQuality:
    name = "D2_retrieval_step_quality"
    def _doc_is_relevant(self, doc_text, supporting_paragraphs):
        doc_lower = doc_text.lower()
        for sup in supporting_paragraphs:
            if sup[:80].lower() in doc_lower: return True
            sup_words = set(sup.lower().split())
            doc_words = set(doc_lower.split())
            if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.7: return True
        return False
    def score(self, trajectory_steps, supporting_paragraphs):
        if not trajectory_steps:
            return {"avg_precision_at_k":0.0, "best_step_precision":0.0, "any_relevant_found":False, "step_precisions":[], "d2_score":0.0}
        step_precisions = []
        any_relevant = False
        for step in trajectory_steps:
            docs = step.get("retrieved_content", [])
            if not docs: step_precisions.append(0.0); continue
            hits = sum(1 for doc in docs if self._doc_is_relevant(doc, supporting_paragraphs))
            prec = hits / len(docs)
            step_precisions.append(round(prec, 3))
            if hits > 0: any_relevant = True
        avg_prec = sum(step_precisions) / len(step_precisions)
        best_prec = max(step_precisions)
        d2_score = round(min(1.0, avg_prec * 0.8 + (0.2 if any_relevant else 0.0)), 3)
        return {"avg_precision_at_k": round(avg_prec,3), "best_step_precision": round(best_prec,3), "any_relevant_found": any_relevant, "step_precisions": step_precisions, "d2_score": d2_score}

class D3_PlanningCoherence:
    name = "D3_planning_coherence"
    def _query_diversity(self, queries):
        if len(queries) <= 1: return 0.0
        unique = list(set(queries))
        if len(unique) == 1: return 0.0
        pairs = []
        for i in range(len(unique)):
            for j in range(i+1, len(unique)):
                pairs.append(1.0 - SequenceMatcher(None, unique[i], unique[j]).ratio())
        return round(sum(pairs)/len(pairs), 3) if pairs else 0.0
    def score(self, trajectory_summary, num_hops):
        unique_queries = trajectory_summary.get("unique_queries", [])
        loop_detected = trajectory_summary.get("loop_detected", False)
        num_unique = len(unique_queries)
        hop_coverage = min(1.0, num_unique / max(num_hops, 1))
        diversity = self._query_diversity(unique_queries)
        loop_penalty = 0.4 if loop_detected else 0.0
        undershoot = max(0.0, (num_hops - num_unique) / max(num_hops, 1))
        undershoot_penalty = undershoot * 0.5
        raw = (hop_coverage * 0.5) + (diversity * 0.5) - loop_penalty - undershoot_penalty
        d3_score = round(max(0.0, min(1.0, raw)), 3)
        return {"num_hops_required": num_hops, "unique_queries_made": num_unique, "hop_coverage": round(hop_coverage,3), "query_diversity": round(diversity,3), "loop_penalty": loop_penalty, "undershoot_penalty": round(undershoot_penalty,3), "d3_score": d3_score}

class D5_TrajectoryEfficiency:
    name = "D5_trajectory_efficiency"
    BASELINE_TOKENS_PER_STEP = 400
    def score(self, trajectory_summary, answer_correct, degenerate=False):
        if degenerate:
            return {"total_steps": 0, "redundancy_rate": 0, "tokens_per_step": 0, "loop_detected": False, "loop_penalty_applied": 0, "efficiency_score": 0.0, "total_tokens": 0, "tokens_per_correct_answer": None, "degenerate": True}
        total_steps = trajectory_summary.get("total_steps", 0)
        redundancy = trajectory_summary.get("redundancy_rate", 0)
        tokens_per_step = trajectory_summary.get("tokens_per_step", 0)
        loop_detected = trajectory_summary.get("loop_detected", False)
        total_tokens = trajectory_summary.get("total_tokens", 0)
        token_penalty = min(1.0, tokens_per_step / (self.BASELINE_TOKENS_PER_STEP * 2))
        loop_penalty = 0.3 if loop_detected else 0.0
        raw = 1.0 - (redundancy*0.5) - (token_penalty*0.3) - loop_penalty
        eff = max(0.0, round(raw, 3))
        return {"total_steps": total_steps, "redundancy_rate": redundancy, "tokens_per_step": tokens_per_step, "loop_detected": loop_detected, "loop_penalty_applied": loop_penalty, "efficiency_score": eff, "total_tokens": total_tokens, "tokens_per_correct_answer": total_tokens if answer_correct else None}

d1_metric = D1_AnswerCorrectness()
d2_metric = D2_RetrievalStepQuality()
d3_metric = D3_PlanningCoherence()
d5_metric = D5_TrajectoryEfficiency()
print("All metric classes ready")

All metric classes ready


---
## Cell 2 — TrajectoryLogger + Rewrite validation + HybridRetriever

All from Week 5.

In [3]:
class TrajectoryLogger:
    def __init__(self, max_steps=10):
        self.max_steps = max_steps
        self.steps = []
        self.loop_detected = False
        self.loop_query = None
    def reset(self):
        self.steps = []; self.loop_detected = False; self.loop_query = None
    def log(self, query, retrieved_docs, latency_ms, supporting_paragraphs=None, rewritten_query=None, rewrite_rejected=False, retrieval_method=None):
        is_loop = bool(self.steps and self.steps[-1]['query'] == query)
        if is_loop: self.loop_detected = True; self.loop_query = query
        precision_at_k = 0.0
        if supporting_paragraphs and retrieved_docs:
            def _is_relevant(doc):
                doc_words = set(doc.lower().split())
                for sup in supporting_paragraphs:
                    sup_words = set(sup.lower().split())
                    if sup_words and len(sup_words & doc_words) / len(sup_words) > 0.5: return True
                return False
            hits = sum(1 for doc in retrieved_docs if _is_relevant(doc))
            precision_at_k = round(hits / len(retrieved_docs), 3)
        tokens_estimate = (len(query) + sum(len(d) for d in retrieved_docs)) // 4
        step = {"step_id": len(self.steps)+1, "query": query, "rewritten_query": rewritten_query, "rewrite_rejected": rewrite_rejected, "retrieval_method": retrieval_method, "num_docs_retrieved": len(retrieved_docs), "retrieved_content": retrieved_docs, "precision_at_k": precision_at_k, "is_loop_step": is_loop, "tokens_estimate": tokens_estimate, "latency_ms": round(latency_ms, 1)}
        self.steps.append(step)
        return step
    def at_limit(self): return len(self.steps) >= self.max_steps
    def summary(self):
        if not self.steps: return {}
        n = len(self.steps)
        unique = list(set(s['query'] for s in self.steps))
        redundant = n - len(unique)
        total_tokens = sum(s['tokens_estimate'] for s in self.steps)
        avg_prec = sum(s['precision_at_k'] for s in self.steps) / n
        return {"total_steps": n, "unique_queries": unique, "redundant_queries": redundant, "redundancy_rate": round(redundant/n, 3), "loop_detected": self.loop_detected, "loop_steps": sum(1 for s in self.steps if s['is_loop_step']), "avg_step_precision_at_k": round(avg_prec, 3), "total_tokens": total_tokens, "total_latency_ms": round(sum(s['latency_ms'] for s in self.steps), 1), "tokens_per_step": round(total_tokens/n, 1)}

# Rewrite validation
BAD_REWRITE_PREFIXES = ["similarity search", "similar vectors", "find locations", "find songs", "find ", "search for", "vector search"]
BAD_REWRITE_CONTAINS = ["->", "similar character", "similar documents", "with similar"]
def validate_rewrite(original_query, rewritten_query):
    rw_lower = rewritten_query.lower().strip()
    for prefix in BAD_REWRITE_PREFIXES:
        if rw_lower.startswith(prefix): return original_query, True
    for pattern in BAD_REWRITE_CONTAINS:
        if pattern in rw_lower: return original_query, True
    if len(rewritten_query) > len(original_query) * 3: return original_query, True
    if len(rewritten_query.strip()) < 3: return original_query, True
    return rewritten_query, False

# Hybrid Retriever
class HybridRetriever:
    RRF_K = 60
    def __init__(self, documents, embedding_model, k=3):
        self.k = k
        self.documents = documents
        docs = [Document(page_content=p) for p in documents]
        self.faiss_store = FAISS.from_documents(docs, embedding_model)
        tokenized = [doc.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized)
    def search_hybrid(self, query):
        faiss_results = self.faiss_store.similarity_search(query, k=self.k)
        faiss_texts = [r.page_content for r in faiss_results]
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        top_bm25_indices = np.argsort(bm25_scores)[::-1][:self.k]
        bm25_texts = [self.documents[i] for i in top_bm25_indices]
        rrf_scores = {}
        for rank, text in enumerate(faiss_texts):
            rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (self.RRF_K + rank + 1)
        for rank, text in enumerate(bm25_texts):
            rrf_scores[text] = rrf_scores.get(text, 0) + 1.0 / (self.RRF_K + rank + 1)
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return [text for text, score in sorted_docs[:self.k]]

print("Loading embedding model...")
embedding_model = OllamaEmbeddings(model="nomic-embed-text")
_ = embedding_model.embed_query("test")
print("TrajectoryLogger + rewrite validation + HybridRetriever ready")

Loading embedding model...
TrajectoryLogger + rewrite validation + HybridRetriever ready


---
## Cell 3 — Load questions + create difficulty variants

From the 50 existing 2-hop questions (RC=2, RD=3), create:
- **1-hop variants (RC=1):** Use decomposition step 2 with step 1's answer as entity.
  Relation→question templates cover all 50 questions. Supporting paragraph = the one containing the answer.
- **Distractor control (RD):** Sample 6/12/18 distractors. Minimum 6 ensures ≥8 total docs for BM25 stability.

This gives 5 conditions × 50 questions = 250 total runs.


In [4]:
q_path = PROJECT_ROOT / "data/questions/3_musique_50.json"
with open(q_path) as f:
    questions_orig = json.load(f)
print(f"Loaded {len(questions_orig)} original 2-hop questions")

# ── Relation-to-question templates ────────────────────────────────
# Expanded to cover ALL relations found in the MuSiQue-50 decompositions.
RELATION_TEMPLATES = {
    "spouse": "Who is the spouse of {entity}?",
    "child": "Who is the child of {entity}?",
    "father": "Who is the father of {entity}?",
    "mother": "Who is the mother of {entity}?",
    "sibling": "Who is the sibling of {entity}?",
    "grandmother": "Who is the grandmother of {entity}?",
    "performer": "Who performed {entity}?",
    "founded by": "Who founded {entity}?",
    "owned by": "What company owns {entity}?",
    "distributed by": "What company distributed {entity}?",
    "manufacturer": "What is the manufacturer of {entity}?",
    "employer": "Who employs {entity}?",
    "headquarters location": "Where is {entity} headquartered?",
    "located in the administrative territorial entity": "What administrative entity is {entity} located in?",
    "located in": "Where is {entity} located?",
    "country": "What country is {entity} in?",
    "county": "What county is {entity} in?",
    "educated at": "Where was {entity} educated?",
    "birthplace": "Where was {entity} born?",
    "place of birth": "Where was {entity} born?",
    "born": "Where was {entity} born?",
    "member of": "What group is {entity} a member of?",
    "record label": "What record label is {entity} on?",
    "award received": "What award did {entity} receive?",
    "genre": "What genre is {entity}?",
    "league": "What league does {entity} play in?",
    "notable work": "What is a notable work by {entity}?",
    "succeeded by": "What succeeded {entity}?",
    "followed by": "What succeeded {entity}?",
    "shares border with": "What shares a border with {entity}?",
    "goal": "What is the goal of {entity}?",
    "movement": "What movement is {entity} part of?",
    "instrument": "What instrument does {entity} play?",
    "capital": "What is the capital of {entity}?",
    "has part": "What is a part of {entity}?",
    "part of": "What is {entity} part of?",
    "instance of": "What type of thing is {entity}?",
    "occupation": "What is the occupation of {entity}?",
    "director": "Who directed {entity}?",
    "author": "Who wrote {entity}?",
    "creator": "Who created {entity}?",
}


def extract_relation_and_entity(sub_q, step1_answer):
    """Extract relation and entity from decomposition step 2.
    
    Handles formats like:
      '#1 >> spouse'           → relation='spouse', entity='Steve Hillage'
      '#1 of france >> mother' → relation='mother', entity='Louis XIII of france'
    """
    parts = sub_q.split('>>')
    relation = parts[-1].strip().lower()
    prefix = '>>'.join(parts[:-1]).strip()
    entity = prefix.replace('#1', step1_answer).strip()
    return relation, entity


def make_1hop_question(decomposition, original_answer):
    """Create a natural-language 1-hop question from decomposition step 2."""
    step2 = decomposition[1]
    step1_answer = decomposition[0]['sub_a']
    
    relation, entity = extract_relation_and_entity(step2['sub_q'], step1_answer)
    
    # Try template match (check all keys, use longest match first)
    best_match = None
    best_len = 0
    for key, template in RELATION_TEMPLATES.items():
        if key in relation and len(key) > best_len:
            best_match = template
            best_len = len(key)
    if best_match:
        return best_match.format(entity=entity)
    
    # Fallback: clean up relation and form a natural question
    clean_rel = relation.replace('#1', '').replace('>>', '').strip()
    return f"What is the {clean_rel} of {entity}?"


def make_rd_variant(supporting_paras, all_distractors, num_distractors, seed):
    """Create a variant KB with controlled distractors.
    
    Args:
        supporting_paras: Supporting paragraphs for this variant
        all_distractors: All available distractor paragraphs
        num_distractors: Target number of distractors
        seed: Random seed for reproducibility
    """
    rng = random.Random(seed)
    if num_distractors >= len(all_distractors):
        selected = all_distractors[:]
    else:
        selected = rng.sample(all_distractors, num_distractors)
    paragraphs = supporting_paras[:] + selected
    rng.shuffle(paragraphs)
    return paragraphs


# ── Build all 5 conditions ────────────────────────────────────────
# Minimum 6 distractors ensures BM25 has enough documents for
# meaningful IDF scores (≥8 total docs in every condition).

CONDITIONS = {
    'A': {'rc': 1, 'rd': 1, 'num_distractors': 6,  'label': 'RC=1, RD=1 (easy)'},
    'B': {'rc': 2, 'rd': 1, 'num_distractors': 6,  'label': 'RC=2, RD=1 (hard reasoning)'},
    'C': {'rc': 1, 'rd': 3, 'num_distractors': 18, 'label': 'RC=1, RD=3 (hard retrieval)'},
    'D': {'rc': 2, 'rd': 2, 'num_distractors': 12, 'label': 'RC=2, RD=2 (medium both)'},
    'E': {'rc': 2, 'rd': 3, 'num_distractors': 18, 'label': 'RC=2, RD=3 (hard both)'},
}

d6_dataset = {}

for cond_name, cond in CONDITIONS.items():
    variants = []
    for q in questions_orig:
        variant = {
            'id': q['id'],
            'original_question': q['question'],
            'condition': cond_name,
            'rc': cond['rc'],
            'rd': cond['rd'],
            'label': cond['label'],
        }
        
        if cond['rc'] == 1:
            # ── 1-hop variant ──
            variant['question'] = make_1hop_question(q['decomposition'], q['answer'])
            variant['answer'] = q['answer']
            variant['num_hops'] = 1
            # Use the supporting paragraph(s) that contain the final answer
            answer_lower = q['answer'].lower()
            sup_1hop = [p for p in q['supporting_paragraphs'] if answer_lower in p.lower()]
            if not sup_1hop:
                sup_1hop = q['supporting_paragraphs'][:1]  # fallback
            variant['supporting_paragraphs'] = sup_1hop
        else:
            # ── 2-hop (original) ──
            variant['question'] = q['question']
            variant['answer'] = q['answer']
            variant['num_hops'] = 2
            variant['supporting_paragraphs'] = q['supporting_paragraphs']
        
        # Build KB with controlled distractors
        variant['paragraphs'] = make_rd_variant(
            variant['supporting_paragraphs'],
            q['distractor_paragraphs'],
            cond['num_distractors'],
            seed=42 + hash(q['id'])
        )
        variant['num_distractors'] = min(cond['num_distractors'], len(q['distractor_paragraphs']))
        
        variants.append(variant)
    d6_dataset[cond_name] = variants

# ── Quality check: detect garbled 1-hop questions ────────────────
print("\n" + "="*65)
print("1-HOP QUESTION QUALITY CHECK")
print("="*65)
bad = []
for v in d6_dataset['A']:
    q_text = v['question']
    if '#1' in q_text or '#2' in q_text or '>>' in q_text:
        bad.append(q_text)
        print(f"  ⚠ GARBLED: {q_text}")
if not bad:
    print("  ✓ All 50 questions are clean natural-language.")
print(f"\nGarbled: {len(bad)}/50")

# ── Show condition examples ──────────────────────────────────────
print("\n" + "="*65)
print("DIFFICULTY VARIANTS CREATED")
print("="*65)
for cond_name, variants in d6_dataset.items():
    print(f"\nCondition {cond_name}: {CONDITIONS[cond_name]['label']}")
    print(f"  Questions: {len(variants)}")
    v0 = variants[0]
    print(f"  Sample Q: {v0['question'][:70]}")
    print(f"  Answer:   {v0['answer']}")
    print(f"  KB size:  {len(v0['paragraphs'])} docs "
          f"({len(v0['supporting_paragraphs'])} supporting + "
          f"{v0['num_distractors']} distractors)")
    print(f"  Hops:     {v0['num_hops']}")

# ── Show a few 1-hop examples side-by-side ───────────────────────
print("\n" + "="*65)
print("1-HOP vs ORIGINAL (first 5)")
print("="*65)
for v in d6_dataset['A'][:5]:
    print(f"  2-hop: {v['original_question']}")
    print(f"  1-hop: {v['question']}")
    print(f"  Answer: {v['answer']}")
    print()

# Save dataset
ds_path = PROJECT_ROOT / "data/questions/6_d6_dataset.json"
with open(ds_path, 'w') as f:
    json.dump(d6_dataset, f, indent=2)
print(f"Saved {sum(len(v) for v in d6_dataset.values())} variants → {ds_path}")


Loaded 50 original 2-hop questions

1-HOP QUESTION QUALITY CHECK
  ✓ All 50 questions are clean natural-language.

Garbled: 0/50

DIFFICULTY VARIANTS CREATED

Condition A: RC=1, RD=1 (easy)
  Questions: 50
  Sample Q: Who is the spouse of Steve Hillage?
  Answer:   Miquette Giraudy
  KB size:  7 docs (1 supporting + 6 distractors)
  Hops:     1

Condition B: RC=2, RD=1 (hard reasoning)
  Questions: 50
  Sample Q: Who is the spouse of the Green performer?
  Answer:   Miquette Giraudy
  KB size:  8 docs (2 supporting + 6 distractors)
  Hops:     2

Condition C: RC=1, RD=3 (hard retrieval)
  Questions: 50
  Sample Q: Who is the spouse of Steve Hillage?
  Answer:   Miquette Giraudy
  KB size:  19 docs (1 supporting + 18 distractors)
  Hops:     1

Condition D: RC=2, RD=2 (medium both)
  Questions: 50
  Sample Q: Who is the spouse of the Green performer?
  Answer:   Miquette Giraudy
  KB size:  14 docs (2 supporting + 12 distractors)
  Hops:     2

Condition E: RC=2, RD=3 (hard both)
  Ques

---
## Cell 4 — Build agent (Week 5 Config C: hybrid + rewrite validation)

In [5]:
TRAJECTORY_LOGGER = TrajectoryLogger(max_steps=10)
current_context = {"retriever": None, "supporting_paragraphs": []}
MAX_CHARS_PER_DOC = 500
llm = ChatOllama(model="llama3.1:8b", temperature=0)

@tool
def lookup_knowledge_base(query: str) -> str:
    """
    Look up information in the knowledge base.
    For multi-hop questions ALWAYS call this tool at least twice.
    Never repeat the same query. If results are unhelpful, change the query.
    """
    if TRAJECTORY_LOGGER.at_limit():
        return "[LOOKUP LIMIT REACHED: provide your best answer now.]"
    rewrite_prompt = ("Rewrite this query to be effective for a vector similarity lookup. "
                      "Extract key entities and facts. Remove filler words. "
                      "Output ONLY the rewritten query, nothing else.\n"
                      f"Original query: {query}")
    try:
        rewritten = llm.invoke([("user", rewrite_prompt)]).content.strip().strip('\"\'')
    except Exception:
        rewritten = query
    rewritten, rewrite_rejected = validate_rewrite(query, rewritten)
    t0 = time.time()
    texts = current_context["retriever"].search_hybrid(rewritten)
    latency = (time.time() - t0) * 1000
    step = TRAJECTORY_LOGGER.log(query, texts, latency, current_context["supporting_paragraphs"],
                                 rewritten_query=rewritten, rewrite_rejected=rewrite_rejected,
                                 retrieval_method="hybrid_bm25_faiss")
    truncated = [t[:MAX_CHARS_PER_DOC] for t in texts]
    warning = "\n\n[You already used this query. Use a DIFFERENT query.]" if step["is_loop_step"] else ""
    nudge = "\n\n[NOTE: You have completed 1 lookup. Multi-hop questions require at least 2 lookups. Look up again with a DIFFERENT query before answering.]" if len(TRAJECTORY_LOGGER.steps) == 1 else ""
    return "\n\n---\n\n".join(truncated) + warning + nudge

SYSTEM_PROMPT = """You are a helpful assistant that answers complex questions.
These questions may need multiple lookups to answer.
RULES:
1. Use the tool at least twice before answering.
2. First lookup: identify the main entity.
3. Second lookup: find the specific fact needed.
4. Never repeat a query — vary your lookups.
5. Give a concise final answer."""

agent = create_react_agent(llm, [lookup_knowledge_base], prompt=SYSTEM_PROMPT)
print("Agent ready — hybrid BM25+FAISS, rewrite validation ON")



Agent ready — hybrid BM25+FAISS, rewrite validation ON


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_95523/4201141364.py:44: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, [lookup_knowledge_base], prompt=SYSTEM_PROMPT)


---
## Cell 5 — Run function

In [6]:
class QuestionTimeoutError(Exception): pass
def _timeout_handler(signum, frame): raise QuestionTimeoutError("Timeout")
QUESTION_TIMEOUT = 120

def run_question(q):
    global current_context
    TRAJECTORY_LOGGER.reset()
    retriever = HybridRetriever(q['paragraphs'], embedding_model, k=3)
    current_context["retriever"] = retriever
    current_context["supporting_paragraphs"] = q['supporting_paragraphs']
    t0 = time.time()
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(QUESTION_TIMEOUT)
    try:
        result = agent.invoke({"messages": [("user", q["question"])]}, config={"recursion_limit": 28})
        answer = result["messages"][-1].content
        success = True
    except QuestionTimeoutError:
        answer = "TIMEOUT"; success = False
    except Exception as e:
        answer = f"ERROR: {e}"; success = False
    finally:
        signal.alarm(0); signal.signal(signal.SIGALRM, old_handler)
    wall_ms = (time.time() - t0) * 1000
    traj_sum = TRAJECTORY_LOGGER.summary()
    d1 = d1_metric.score(answer, q["answer"])
    d2 = d2_metric.score(TRAJECTORY_LOGGER.steps, q["supporting_paragraphs"])
    d3 = d3_metric.score(traj_sum, q["num_hops"])
    d5 = d5_metric.score(traj_sum, d1["exact_match"], degenerate=d1.get("degenerate_output", False))
    return {
        "question_id": q["id"], "question": q["question"], "ground_truth": q["answer"],
        "num_hops": q["num_hops"], "condition": q["condition"], "rc": q["rc"], "rd": q["rd"],
        "predicted_answer": answer, "success": success, "wall_latency_ms": round(wall_ms, 1),
        "trajectory": TRAJECTORY_LOGGER.steps.copy(), "trajectory_summary": traj_sum,
        "D1_answer_correctness": d1, "D2_retrieval_step_quality": d2,
        "D3_planning_coherence": d3, "D5_trajectory_efficiency": d5,
    }
print("Run function ready")

Run function ready


---
## Cell 6 — Run all 5 conditions

5 conditions × 50 questions ≈ 80–90 minutes total. Results saved incrementally.

In [7]:
all_results = {}  # condition -> list of result dicts
out_path = PROJECT_ROOT / "data/trajectories/6_d6_results.json"

for cond_name in ['A', 'B', 'C', 'D', 'E']:
    cond = CONDITIONS[cond_name]
    variants = d6_dataset[cond_name]
    
    print(f"\n{'='*65}")
    print(f"CONDITION {cond_name}: {cond['label']}")
    print(f"{'='*65}\n")
    
    results = []
    for i, q in enumerate(variants):
        print(f"  [{cond_name}] Q{i+1}/50: {q['question'][:55]}...")
        r = run_question(q)
        results.append(r)
        d1s = r['D1_answer_correctness']['score']
        d2s = r['D2_retrieval_step_quality']['d2_score']
        wall = r['wall_latency_ms'] / 1000
        tag = "✓" if r['D1_answer_correctness']['exact_match'] else "✗"
        print(f"    {tag} D1={d1s:.3f}  D2={d2s:.3f}  ({wall:.0f}s)")
    
    all_results[cond_name] = results
    correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
    print(f"\n  → Condition {cond_name}: {correct}/50 = {correct/50:.0%}")
    
    # Save after each condition
    with open(out_path, 'w') as f:
        json.dump(all_results, f, indent=2)

print(f"\n{'='*65}")
print("ALL CONDITIONS COMPLETE")
print(f"{'='*65}")
for cond_name, results in all_results.items():
    correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
    avg_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in results) / len(results)
    print(f"  {cond_name} ({CONDITIONS[cond_name]['label']:30}): {correct}/50 = {correct/50:5.0%}  avg D2={avg_d2:.3f}")
print(f"\nSaved to {out_path}")


CONDITION A: RC=1, RD=1 (easy)

  [A] Q1/50: Who is the spouse of Steve Hillage?...
    ✓ D1=1.000  D2=0.466  (21s)
  [A] Q2/50: Who founded Orion Pictures?...
    ✗ D1=0.091  D2=0.466  (26s)
  [A] Q3/50: What administrative entity is Nuevo Laredo located in?...
    ✗ D1=0.000  D2=0.466  (8s)
  [A] Q4/50: Where is German Aerospace Center headquartered?...
    ✓ D1=1.000  D2=0.333  (13s)
  [A] Q5/50: What company owns Bombardier Aerospace?...
    ✓ D1=1.000  D2=0.466  (22s)
  [A] Q6/50: Who is the child of Daniel Webster?...
    ✓ D1=1.000  D2=0.466  (22s)
  [A] Q7/50: Who is the mother of Louis XIII of France?...
    ✓ D1=1.000  D2=0.466  (31s)
  [A] Q8/50: What movement is European Movement International part o...
    ✗ D1=0.200  D2=0.466  (21s)
  [A] Q9/50: What succeeded Adelphia Communications Corporation?...
    ✓ D1=1.000  D2=0.333  (35s)
  [A] Q10/50: What shares a border with Niassa Province?...
    ✓ D1=1.000  D2=0.466  (46s)
  [A] Q11/50: What league does Benevento Calcio pl

---
## Cell 7 — D6 metric: compute interaction effect

In [8]:
# ── Compute per-condition stats ──────────────────────────────────

def condition_stats(results):
    n = len(results)
    correct = sum(1 for r in results if r['D1_answer_correctness']['exact_match'])
    nd = [r for r in results if not r['D1_answer_correctness'].get('degenerate_output', False)]
    avg_d1 = sum(r['D1_answer_correctness']['score'] for r in results) / n
    avg_d2 = sum(r['D2_retrieval_step_quality']['d2_score'] for r in nd) / len(nd) if nd else 0
    avg_d3 = sum(r['D3_planning_coherence']['d3_score'] for r in nd) / len(nd) if nd else 0
    d2_zero = sum(1 for r in nd if r['D2_retrieval_step_quality']['d2_score'] == 0)
    return {'n': n, 'correct': correct, 'accuracy': correct/n, 'avg_d1': avg_d1, 'avg_d2': avg_d2, 'avg_d3': avg_d3, 'd2_zero': d2_zero}

stats = {}
for cond_name, results in all_results.items():
    stats[cond_name] = condition_stats(results)

# ── D6 interaction effect ────────────────────────────────────────
# Additive prediction: if RC and RD were independent,
# accuracy(RC=2, RD=3) ≈ accuracy(A) - drop_from_RC - drop_from_RD
#   where drop_from_RC = accuracy(A) - accuracy(B)   [going from RC=1 to RC=2, holding RD=1]
#   and   drop_from_RD = accuracy(A) - accuracy(C)   [going from RD=1 to RD=3, holding RC=1]

acc_A = stats['A']['accuracy']  # baseline (easy)
acc_B = stats['B']['accuracy']  # hard RC only
acc_C = stats['C']['accuracy']  # hard RD only
acc_E = stats['E']['accuracy']  # hard both (actual)

drop_RC = acc_A - acc_B  # accuracy loss from adding hard reasoning
drop_RD = acc_A - acc_C  # accuracy loss from adding hard retrieval
predicted_additive = acc_A - drop_RC - drop_RD  # if independent
actual = acc_E

interaction_effect = predicted_additive - actual  # positive = super-linear collapse

print("D6 INTERACTION ANALYSIS")
print("="*50)
print(f"Baseline (A: RC=1, RD=1):        {acc_A:.0%}")
print(f"Hard reasoning only (B: RC=2):   {acc_B:.0%}  (drop: {drop_RC:+.0%})")
print(f"Hard retrieval only (C: RD=3):   {acc_C:.0%}  (drop: {drop_RD:+.0%})")
print(f"")
print(f"If additive (independent):       {predicted_additive:.0%}")
print(f"Actual (E: RC=2, RD=3):          {actual:.0%}")
print(f"Interaction effect:              {interaction_effect:+.0%}")
print(f"")
if interaction_effect > 0.05:
    print("→ SUPER-LINEAR COLLAPSE CONFIRMED")
    print(f"  The combined difficulty is {interaction_effect:.0%} worse than the sum of individual effects.")
    print(f"  This proves D6: difficulty axes interact, not just add.")
elif interaction_effect < -0.05:
    print("→ SUB-LINEAR: difficulties are partially redundant")
else:
    print("→ APPROXIMATELY ADDITIVE: difficulties are independent")

# Save D6 summary
d6_summary = {
    'stats': {k: {kk: vv for kk, vv in v.items()} for k, v in stats.items()},
    'interaction': {
        'baseline_accuracy': acc_A,
        'drop_from_RC': drop_RC,
        'drop_from_RD': drop_RD,
        'predicted_additive': predicted_additive,
        'actual_combined': actual,
        'interaction_effect': interaction_effect,
        'super_linear': interaction_effect > 0.05,
    }
}
d6_path = PROJECT_ROOT / "data/trajectories/6_d6_summary.json"
with open(d6_path, 'w') as f:
    json.dump(d6_summary, f, indent=2)
print(f"\nSaved D6 summary to {d6_path}")

D6 INTERACTION ANALYSIS
Baseline (A: RC=1, RD=1):        74%
Hard reasoning only (B: RC=2):   48%  (drop: +26%)
Hard retrieval only (C: RD=3):   64%  (drop: +10%)

If additive (independent):       38%
Actual (E: RC=2, RD=3):          24%
Interaction effect:              +14%

→ SUPER-LINEAR COLLAPSE CONFIRMED
  The combined difficulty is 14% worse than the sum of individual effects.
  This proves D6: difficulty axes interact, not just add.

Saved D6 summary to /Users/idhantsingh/Desktop/agenticrag-bench/data/trajectories/6_d6_summary.json


---
## Cell 8 — Publication-quality plots

4-panel figure: degradation curve, heatmap, D2 by condition, interaction effect.

In [9]:
# ── Style setup ──────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.alpha': 0.6,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
})

# Color palette
COLORS = {
    'A': '#3fb950',  # green
    'B': '#58a6ff',  # blue
    'C': '#d29922',  # amber
    'D': '#bc8cff',  # purple
    'E': '#f85149',  # red
}
GRADIENT_CMAP = mcolors.LinearSegmentedColormap.from_list('accuracy', ['#f85149', '#d29922', '#3fb950'])

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle('D6 — Difficulty Interaction Collapse', fontsize=20, fontweight='bold', color='white', y=0.98)
fig.text(0.5, 0.945, 'AgenticRAG-Bench · LangGraph ReAct + Llama 3.1 8B · 50 MuSiQue questions per condition',
         ha='center', fontsize=10, color='#8b949e')

# ── Panel 1: Accuracy Degradation Curve ──────────────────────────
ax1 = axes[0, 0]
ax1.set_title('Accuracy vs Number of Hard Axes', pad=12)

# Group by number of hard axes
hard_axes_map = {
    'A': 0,  # RC=1, RD=1 → 0 hard axes
    'B': 1,  # RC=2, RD=1 → 1 hard axis (reasoning)
    'C': 1,  # RC=1, RD=3 → 1 hard axis (retrieval)
    'D': 2,  # RC=2, RD=2 → 1.5 (we'll plot as 1.5 or 2)
    'E': 2,  # RC=2, RD=3 → 2 hard axes
}

# Plot individual conditions as scatter
for cond_name in ['A', 'B', 'C', 'D', 'E']:
    x = hard_axes_map[cond_name]
    y = stats[cond_name]['accuracy'] * 100
    ax1.scatter(x, y, c=COLORS[cond_name], s=180, zorder=5, edgecolors='white', linewidths=1.5)
    label_offset = (0.08, 3) if cond_name not in ['B', 'C'] else (0.08, -5)
    ax1.annotate(f'{cond_name}\n{y:.0f}%', (x, y), xytext=(x + label_offset[0], y + label_offset[1]),
                fontsize=9, fontweight='bold', color=COLORS[cond_name])

# Linear prediction line
x_linear = [0, 1, 2]
y_linear = [acc_A * 100, acc_A * 100 - (drop_RC + drop_RD) / 2 * 100, predicted_additive * 100]
ax1.plot(x_linear, y_linear, '--', color='#8b949e', alpha=0.7, linewidth=1.5, label='Linear prediction')

# Actual degradation line (A → avg(B,C) → E)
avg_1axis = (stats['B']['accuracy'] + stats['C']['accuracy']) / 2 * 100
x_actual = [0, 1, 2]
y_actual = [acc_A * 100, avg_1axis, acc_E * 100]
ax1.plot(x_actual, y_actual, '-', color='#f85149', linewidth=2.5, label='Actual', zorder=4)

# Shade the gap
ax1.fill_between(x_actual, y_actual, y_linear, alpha=0.15, color='#f85149')
mid_y = (y_linear[2] + y_actual[2]) / 2
if abs(y_linear[2] - y_actual[2]) > 3:
    ax1.annotate(f'Interaction\neffect: {interaction_effect*100:+.0f}pp',
                xy=(2, mid_y), fontsize=9, color='#f85149', fontweight='bold',
                ha='left', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#161b22', edgecolor='#f85149', alpha=0.9))

ax1.set_xlabel('Number of hard difficulty axes')
ax1.set_ylabel('Accuracy (%)')
ax1.set_xticks([0, 1, 2])
ax1.set_xticklabels(['0\n(easy)', '1\n(single axis)', '2\n(compound)'])
ax1.set_ylim(0, 105)
ax1.legend(loc='upper right', fontsize=9, framealpha=0.8)
ax1.grid(True, alpha=0.3)

# ── Panel 2: RC × RD Heatmap ────────────────────────────────────
ax2 = axes[0, 1]
ax2.set_title('Accuracy Heatmap: RC × RD', pad=12)

# Build 2×3 grid: RC (1,2) × RD (1,2,3)
# Map conditions to grid positions
grid = np.full((2, 3), np.nan)  # RC rows × RD cols
grid_labels = [['' for _ in range(3)] for _ in range(2)]
cond_grid = {
    (0, 0): 'A',  # RC=1, RD=1
    (1, 0): 'B',  # RC=2, RD=1
    (0, 2): 'C',  # RC=1, RD=3
    (1, 1): 'D',  # RC=2, RD=2
    (1, 2): 'E',  # RC=2, RD=3
}
for (r, c), cond_name in cond_grid.items():
    grid[r, c] = stats[cond_name]['accuracy'] * 100
    grid_labels[r][c] = f"{cond_name}\n{grid[r,c]:.0f}%"

im = ax2.imshow(grid, cmap=GRADIENT_CMAP, vmin=0, vmax=100, aspect='auto', interpolation='nearest')

# Annotate cells
for r in range(2):
    for c in range(3):
        if not np.isnan(grid[r, c]):
            text_color = 'white' if grid[r, c] < 50 else '#0d1117'
            ax2.text(c, r, grid_labels[r][c], ha='center', va='center', fontsize=13, fontweight='bold', color=text_color)
        else:
            ax2.text(c, r, '—', ha='center', va='center', fontsize=13, color='#8b949e')

ax2.set_xticks([0, 1, 2])
ax2.set_xticklabels(['RD=1\n(2 distractors)', 'RD=2\n(8 distractors)', 'RD=3\n(18 distractors)'])
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['RC=1\n(1-hop)', 'RC=2\n(2-hop)'])
ax2.set_xlabel('Retrieval Difficulty')
ax2.set_ylabel('Reasoning Complexity')

# ── Panel 3: D2 by Condition ────────────────────────────────────
ax3 = axes[1, 0]
ax3.set_title('Avg D2 (Retrieval Quality) by Condition', pad=12)

conds = ['A', 'B', 'C', 'D', 'E']
d2_vals = [stats[c]['avg_d2'] for c in conds]
bars = ax3.bar(conds, d2_vals, color=[COLORS[c] for c in conds], edgecolor='white', linewidth=0.8, width=0.6)

for bar, val, cond in zip(bars, d2_vals, conds):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015, f'{val:.3f}',
            ha='center', fontsize=10, fontweight='bold', color=COLORS[cond])

ax3.set_ylabel('Avg D2 Score')
ax3.set_xlabel('Condition')
ax3.set_ylim(0, max(d2_vals) * 1.25)
ax3.grid(True, axis='y', alpha=0.3)

# Add condition labels below
for i, c in enumerate(conds):
    ax3.text(i, -0.06 * max(d2_vals) * 1.25, CONDITIONS[c]['label'].split('(')[1].rstrip(')'),
            ha='center', fontsize=8, color='#8b949e', transform=ax3.get_xaxis_transform())

# ── Panel 4: Interaction Effect ──────────────────────────────────
ax4 = axes[1, 1]
ax4.set_title('D6 Interaction Effect', pad=12)

categories = ['Baseline\n(A: easy)', 'Drop from\nhard RC', 'Drop from\nhard RD', 'Predicted\n(additive)', 'Actual\n(E: hard both)']
values = [acc_A * 100, -drop_RC * 100, -drop_RD * 100, predicted_additive * 100, actual * 100]
bar_colors = ['#3fb950', '#58a6ff', '#d29922', '#8b949e', '#f85149']

# Waterfall-style: show baseline, then drops, then comparison
# Simpler: just show predicted vs actual as two big bars
comparison_labels = ['If independent\n(additive)', 'Actual\n(combined)']
comparison_vals = [predicted_additive * 100, actual * 100]
comparison_colors = ['#8b949e', '#f85149']

bars = ax4.bar(comparison_labels, comparison_vals, color=comparison_colors, edgecolor='white',
              linewidth=0.8, width=0.5)

for bar, val in zip(bars, comparison_vals):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, f'{val:.0f}%',
            ha='center', fontsize=14, fontweight='bold', color='white')

# Draw the gap arrow
if abs(predicted_additive - actual) > 0.02:
    gap_pct = interaction_effect * 100
    mid = (comparison_vals[0] + comparison_vals[1]) / 2
    ax4.annotate('', xy=(1, comparison_vals[1]), xytext=(1, comparison_vals[0]),
                arrowprops=dict(arrowstyle='<->', color='#f0883e', lw=2.5))
    ax4.text(1.3, mid, f'{gap_pct:+.0f}pp\ninteraction\neffect',
            fontsize=11, fontweight='bold', color='#f0883e', va='center')

# Add context bars (smaller, semi-transparent)
context_x = [-0.7, -0.3, 0.1]
context_vals = [acc_A * 100, acc_B * 100, acc_C * 100]
context_labels = ['A', 'B', 'C']
for x, val, label, color in zip(context_x, context_vals, context_labels, ['#3fb950', '#58a6ff', '#d29922']):
    ax4.bar(x, val, width=0.15, color=color, alpha=0.4, edgecolor=color, linewidth=0.5)
    ax4.text(x, val + 1.5, f'{label}:{val:.0f}%', ha='center', fontsize=7, color=color)

ax4.set_ylabel('Accuracy (%)')
ax4.set_ylim(0, 105)
ax4.grid(True, axis='y', alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.93])

plot_path = PROJECT_ROOT / "notes/6_d6_plots.png"
plt.savefig(plot_path, dpi=200, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f"Plots saved to {plot_path}")

Plots saved to /Users/idhantsingh/Desktop/agenticrag-bench/notes/6_d6_plots.png


/var/folders/tl/vfsv8k5j2bv967f2r2x0ph7m0000gn/T/ipykernel_95523/4267974288.py:188: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Cell 9 — Auto-generate observations

In [10]:
obs = f"""# Week 6 Observations — AgenticRAG-Bench
Date: {time.strftime('%Y-%m-%d')}

## What was built
- D6 Difficulty Interaction Collapse: 5 controlled difficulty conditions from the same 50 questions
- 1-hop variants created via decomposition (template-based question generation)
- Distractor count controlled: RD=1 (2), RD=2 (8), RD=3 (18)
- Publication-quality 4-panel visualization

## Results by condition

| Condition | RC | RD | Accuracy | Avg D2 | D2=0 | Avg D3 |
|---|---|---|---|---|---|---|
| A (easy)           | 1 | 1 | {stats['A']['correct']}/50 = {stats['A']['accuracy']:.0%} | {stats['A']['avg_d2']:.3f} | {stats['A']['d2_zero']} | {stats['A']['avg_d3']:.3f} |
| B (hard reasoning) | 2 | 1 | {stats['B']['correct']}/50 = {stats['B']['accuracy']:.0%} | {stats['B']['avg_d2']:.3f} | {stats['B']['d2_zero']} | {stats['B']['avg_d3']:.3f} |
| C (hard retrieval)  | 1 | 3 | {stats['C']['correct']}/50 = {stats['C']['accuracy']:.0%} | {stats['C']['avg_d2']:.3f} | {stats['C']['d2_zero']} | {stats['C']['avg_d3']:.3f} |
| D (medium both)    | 2 | 2 | {stats['D']['correct']}/50 = {stats['D']['accuracy']:.0%} | {stats['D']['avg_d2']:.3f} | {stats['D']['d2_zero']} | {stats['D']['avg_d3']:.3f} |
| E (hard both)      | 2 | 3 | {stats['E']['correct']}/50 = {stats['E']['accuracy']:.0%} | {stats['E']['avg_d2']:.3f} | {stats['E']['d2_zero']} | {stats['E']['avg_d3']:.3f} |

## D6 interaction analysis
- Baseline accuracy (A):        {stats['A']['accuracy']:.0%}
- Drop from hard RC (A→B):      {drop_RC:+.0%}
- Drop from hard RD (A→C):      {drop_RD:+.0%}
- Predicted if additive:        {predicted_additive:.0%}
- Actual combined (E):          {actual:.0%}
- Interaction effect:           {interaction_effect:+.0%}
- Super-linear collapse:        {'YES' if interaction_effect > 0.05 else 'NO'}

## Key observations
- TBD after reviewing results and plots

## Week 7 plan
- TBD
"""

obs_path = PROJECT_ROOT / "notes/6_observations.md"
with open(obs_path, 'w') as f:
    f.write(obs)
print(f"Observations saved to {obs_path}")
print("\n" + obs)

Observations saved to /Users/idhantsingh/Desktop/agenticrag-bench/notes/6_observations.md

# Week 6 Observations — AgenticRAG-Bench
Date: 2026-06-21

## What was built
- D6 Difficulty Interaction Collapse: 5 controlled difficulty conditions from the same 50 questions
- 1-hop variants created via decomposition (template-based question generation)
- Distractor count controlled: RD=1 (2), RD=2 (8), RD=3 (18)
- Publication-quality 4-panel visualization

## Results by condition

| Condition | RC | RD | Accuracy | Avg D2 | D2=0 | Avg D3 |
|---|---|---|---|---|---|---|
| A (easy)           | 1 | 1 | 37/50 = 74% | 0.414 | 3 | 0.599 |
| B (hard reasoning) | 2 | 1 | 24/50 = 48% | 0.399 | 15 | 0.520 |
| C (hard retrieval)  | 1 | 3 | 32/50 = 64% | 0.354 | 10 | 0.601 |
| D (medium both)    | 2 | 2 | 15/50 = 30% | 0.330 | 17 | 0.498 |
| E (hard both)      | 2 | 3 | 12/50 = 24% | 0.278 | 20 | 0.494 |

## D6 interaction analysis
- Baseline accuracy (A):        74%
- Drop from hard RC (A→B):      +26%
